# > Initial setup

In [ ]:
!pip install bert-score anthropic rouge_score

In [ ]:
from google.colab import userdata
api_key = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic(api_key=api_key)

In [2]:
AI_GENERATED = """
Chart 1: Scoreboard Overview
L2: Sales £733,215, profit £93,439, returns 4,655 units, quantity 12,476.
L3: Sales/quantity grow YoY but returns rise too.
L4: Monitor returns to protect margins amid expansion.
Chart 2: 2023 | Sales vs Targets
L2: Sales hit £733K, exceeding the target of £715K by +20.4% vs PY.
L3: All categories meet or exceed targets except for Phones and Binders.
L4: Focus on maintaining consistency in growth across all segments.
Chart 3: Segment | Sales vs Targets
L2: Consumer sales are down -£24.3K, Corporate -£6.7K, Home Office +£33.2K.
L3: Home Office shows strong performance.
L4: Consider increasing marketing efforts for the Home Office segment.
Chart 4: Top 5 Sub-Categories | Sales vs Targets
L2: Phones lead with £10.6K sales, followed by Chairs at £5.1K and Binders at £0.9K.
L3: The top three sub-categories are all in positive growth.
L4: Continue to invest in product development for the most profitable categories.
Chart 5: Sales by Location | Top 5 States
L2: California leads with £146,388, New York at £93,923, Washington at £65,540, Texas at £43,422, and Pennsylvania at £42,688.
L3: The top states are concentrated in the West.
L4: Target expansion into high-value markets like California.
Chart 6: Top 5 Manufacturers | Sales vs Targets
L2: Canon leads with +£6.1K sales, Global -£3.9K, Hon -£11.3K, GBC +£6.8K, Fellowes +£9.8K.
L3: The top three are all in positive growth.
L4: Focus on maintaining strong performance from the top manufacturers.
"""

In [3]:
HUMAN_GENERATED = """
Chart 1: Scoreboard overview
L2: Sales are about 733K, profit is 93K, returns are around 4.7K, and quantity is about 12.5K units, each showing a noticeable yoy change.
L3: Overall, the business is growing in sales and quantity with generally improving profit.
L4: However, while the company is successfully expanding revenue and volume, the return rates should be monitored to ensure growth does not erode profitability or customer satisfaction.

Chart 2: 2023 | Sales vs Targets
L2: Sales started at around 40K in January and ended the year at around 80K, but it did not reach the target.
L3: The chart shows green circles for January, June, August, October, and November, which reached the target.
L4: In conclusion, sales improved over time, with high‑performing later months helping lift weaker early‑year periods in the next cycle.

Chart 3: Segment | Sales vs Targets
L2: Consumer and Corporate segments sit below their targets.
L3: The Home Office exceeds its target by 33.2K.
L4: Strategy should probe what is driving Home Office success and apply its tactics to Consumer and Corporate segments to close their gaps.

Chart 4: Top 5 Sub-Categories | Sales vs Targets
L2: Phones lead the sub-categories sales with +10.6K sales
L3: Phones, binders, and copiers have exceeded their targets, while the chairs lagged behind by 5.1K, and storage was just 0.9K below its target.
L4: The target should be adjusted for chairs as it is higher than the phones' target, which phones are hot commodity in these days

Chart 5: Sales by Location | Top 5 States
L2: California leads the sales by 146K, New York, Washington, Texas, and Pennsylvania follow with progressively smaller yet still substantial figures.
L3: A small group of states contributes a disproportionate share of total sales, with coastal and large‑population states dominating the top five.
L4: This geographic condition suggests expansion into mid-tier states could be the next growth target.

Chart 6: Top 5 Manufacturers chart | Sales vs Targets
L2: Canon leads with +6.1K sales
L3: Canon, GBC, and Fellowes have met their targets, while the Global and Hon manufacturers have lagged behind theirs.
L4: Even though the Global and Hon have their sales ranked in 2nd and 3rd place, the target is too high, should lower the target
"""

# > Evaluation

## 1. ROUGE

In [4]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"], use_stemmer=True
)
scores = scorer.score(HUMAN_GENERATED, AI_GENERATED)

print("ROUGE SCORES")
print("=" * 40)
for metric, result in scores.items():
    print(f"\n{metric.upper()}")
    print(f"  Precision : {result.precision:.4f}")
    print(f"  Recall    : {result.recall:.4f}")
    print(f"  F1        : {result.fmeasure:.4f}")

ROUGE SCORES

ROUGE1
  Precision : 0.6409
  Recall    : 0.4323
  F1        : 0.5163

ROUGE2
  Precision : 0.2829
  Recall    : 0.1906
  F1        : 0.2278

ROUGEL
  Precision : 0.4517
  Recall    : 0.3047
  F1        : 0.3639


## 2. BERTScore

In [ ]:
import torch
from bert_score import score as bert_score

P, R, F1 = bert_score(
    [AI_GENERATED],
    [HUMAN_GENERATED],
    lang="en",
    model_type="distilbert-base-uncased",
    num_layers=5,
    verbose=True
)

print("\nBERTScore")
print("=" * 40)
print(f"  Precision : {P[0].item():.4f}")
print(f"  Recall    : {R[0].item():.4f}")
print(f"  F1        : {F1[0].item():.4f}")

Using device: mps


## 3. G-Eval (LLM-as-Judge)

In [ ]:
# G-Eval scores 4 dimensions (1–5 scale each):
#   Factual Accuracy  – do the numbers match the reference?
#   Completeness      – are all charts covered?
#   Analytical Depth  – does L3/L4 go beyond surface-level facts?
#   Conciseness       – is language crisp and executive-appropriate?

import os, json
import anthropic
from google.colab import userdata

GEVAL_PROMPT = """You are an expert evaluator for BI dashboard insight quality.

You will be given:
- REFERENCE: a human-written expert insight (ground truth)
- CANDIDATE: an AI-generated insight to evaluate

Score the CANDIDATE on these 4 dimensions, each from 1 (poor) to 5 (excellent):
1. Factual Accuracy: Do the numbers, signs, and comparisons match the REFERENCE?
2. Completeness: Does the CANDIDATE cover all charts/sections in the REFERENCE?
3. Analytical Depth: Does the CANDIDATE identify patterns and implications (L3/L4), not just restate facts?
4. Conciseness: Is the language crisp and executive-appropriate, without padding or vague filler?

Respond ONLY with valid JSON in this exact format:
{{
  "factual_accuracy": <1-5>,
  "completeness": <1-5>,
  "analytical_depth": <1-5>,
  "conciseness": <1-5>,
  "overall": <average of the four, rounded to 2 decimal places>,
  "rationale": "<2-3 sentences explaining the scores>"
}}

REFERENCE:
{reference}

CANDIDATE:
{candidate}
"""

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

message = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=512,
    messages=[{
        "role": "user",
        "content": GEVAL_PROMPT.format(
            reference=HUMAN_GENERATED,
            candidate=AI_GENERATED
        )
    }]
)

raw = message.content[0].text.strip()
if raw.startswith("```"):
    raw = raw.split("```")[1]
    if raw.startswith("json"):
        raw = raw[4:]
    raw = raw.strip()

result = json.loads(raw)

print("G-EVAL SCORES")
print("=" * 40)
print(f"  Factual Accuracy  : {result['factual_accuracy']} / 5")
print(f"  Completeness      : {result['completeness']} / 5")
print(f"  Analytical Depth  : {result['analytical_depth']} / 5")
print(f"  Conciseness       : {result['conciseness']} / 5")
print(f"  Overall           : {result['overall']} / 5")
print(f"\n  Rationale: {result['rationale']}")

# > Evaluate per-chart-per-level design

## Parse texts

In [ ]:
import re

def parse_charts(text):
    """
    Returns a list of dicts:
    [{ "chart_id": 1, "title": "...", "L2": "...", "L3": "...", "L4": "..." }, ...]
    """
    charts = []
    # Split on "Chart N:" pattern
    blocks = re.split(r"(?=Chart\s+\d+[:.])", text.strip())
    for block in blocks:
        block = block.strip()
        if not block:
            continue
        lines = block.splitlines()
        # Extract chart header
        header = lines[0].strip()
        match = re.match(r"Chart\s+(\d+)[:.]\s*(.*)", header)
        if not match:
            continue
        chart_id = int(match.group(1))
        title    = match.group(2).strip()
        L2 = L3 = L4 = ""
        for line in lines[1:]:
            line = line.strip()
            if line.startswith("L2:"):
                L2 = line[3:].strip()
            elif line.startswith("L3:"):
                L3 = line[3:].strip()
            elif line.startswith("L4:"):
                L4 = line[3:].strip()
        charts.append({
            "chart_id": chart_id,
            "title":    title,
            "L2":       L2,
            "L3":       L3,
            "L4":       L4,
        })
    return charts

ai_charts    = parse_charts(AI_GENERATED)
human_charts = parse_charts(HUMAN_GENERATED)

# Align by chart_id — only score charts present in both
ai_ids    = {c["chart_id"]: c for c in ai_charts}
human_ids = {c["chart_id"]: c for c in human_charts}
shared_ids = sorted(set(ai_ids.keys()) & set(human_ids.keys()))

print(f"✅ Parsed: {len(ai_charts)} AI charts | {len(human_charts)} human charts")
print(f"   Matched chart IDs: {shared_ids}")
for cid in shared_ids:
    print(f"\n  Chart {cid}: {ai_ids[cid]['title']}")
    print(f"    AI    L2: {ai_ids[cid]['L2'][:60]}...")
    print(f"    Human L2: {human_ids[cid]['L2'][:60]}...")

## 1. ROUGE per-chart-per-level

In [ ]:
from rouge_score import rouge_scorer
import numpy as np

rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def rouge_f1(hypothesis, reference, metric):
    if not hypothesis.strip() or not reference.strip():
        return None
    return round(rouge.score(reference, hypothesis)[metric].fmeasure, 4)

# ── Score every chart × every level ─────────────────────────
rouge_rows = []

print("ROUGE PER CHART PER LEVEL")
print("=" * 70)
print(f"{'Chart':<35} {'L2-R1':>6} {'L3-R1':>6} {'L4-R1':>6} {'L2-RL':>6} {'L3-RL':>6} {'L4-RL':>6}")
print("-" * 70)

for cid in shared_ids:
    ai_c    = ai_ids[cid]
    human_c = human_ids[cid]
    row = {
        "chart_id":  cid,
        "title":     ai_c["title"],
        "L2_rouge1": rouge_f1(ai_c["L2"], human_c["L2"], "rouge1"),
        "L2_rougeL": rouge_f1(ai_c["L2"], human_c["L2"], "rougeL"),
        "L3_rouge1": rouge_f1(ai_c["L3"], human_c["L3"], "rouge1"),
        "L3_rougeL": rouge_f1(ai_c["L3"], human_c["L3"], "rougeL"),
        "L4_rouge1": rouge_f1(ai_c["L4"], human_c["L4"], "rouge1"),
        "L4_rougeL": rouge_f1(ai_c["L4"], human_c["L4"], "rougeL"),
    }
    rouge_rows.append(row)
    print(f"  {row['title'][:33]:<33} "
          f"  {str(row['L2_rouge1']):>6}"
          f"  {str(row['L3_rouge1']):>6}"
          f"  {str(row['L4_rouge1']):>6}"
          f"  {str(row['L2_rougeL']):>6}"
          f"  {str(row['L3_rougeL']):>6}"
          f"  {str(row['L4_rougeL']):>6}")

# ── Dashboard-level rollup ────────────────────────────────────
print("\nDASHBOARD AVERAGES (ROUGE-1 F1)")
print("-" * 40)
for level in ["L2", "L3", "L4"]:
    vals = [r[f"{level}_rouge1"] for r in rouge_rows if r[f"{level}_rouge1"] is not None]
    print(f"  {level} : {np.mean(vals):.4f}")

rouge_dashboard = {
    level: round(np.mean([r[f"{level}_rouge1"] for r in rouge_rows
                          if r[f"{level}_rouge1"] is not None]), 4)
    for level in ["L2", "L3", "L4"]
}
rouge_dashboard["overall"] = round(np.mean(list(rouge_dashboard.values())), 4)
print(f"  Overall : {rouge_dashboard['overall']:.4f}")

## 2. BERTScore per-chart-per-level

In [ ]:
import torch
from bert_score import score as bert_score_fn
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}\n")

def bertscore_batch(hypotheses, references):
    """Score a list of (hypothesis, reference) pairs in one batch call."""
    if not hypotheses:
        return []
    P, R, F1 = bert_score_fn(
        hypotheses, references,
        lang="en",
        model_type="distilbert-base-uncased",
        num_layers=5,
        device=device,
        verbose=False
    )
    return [round(f.item(), 4) for f in F1]

# ── Batch all charts per level for efficiency ────────────────
bert_rows = {cid: {"chart_id": cid, "title": ai_ids[cid]["title"]}
             for cid in shared_ids}

for level in ["L2", "L3", "L4"]:
    hyps = [ai_ids[cid][level]    for cid in shared_ids]
    refs = [human_ids[cid][level] for cid in shared_ids]
    scores = bertscore_batch(hyps, refs)
    for cid, score in zip(shared_ids, scores):
        bert_rows[cid][f"{level}_bert_f1"] = score

bert_rows = list(bert_rows.values())

# ── Print ─────────────────────────────────────────────────────
print("BERTScore PER CHART PER LEVEL (F1)")
print("=" * 60)
print(f"{'Chart':<35} {'L2':>6} {'L3':>6} {'L4':>6}")
print("-" * 60)
for row in bert_rows:
    print(f"  {row['title'][:33]:<33} "
          f"  {row['L2_bert_f1']:>6}"
          f"  {row['L3_bert_f1']:>6}"
          f"  {row['L4_bert_f1']:>6}")

# ── Dashboard-level rollup ────────────────────────────────────
print("\nDASHBOARD AVERAGES (BERTScore F1)")
print("-" * 40)
bert_dashboard = {}
for level in ["L2", "L3", "L4"]:
    vals = [r[f"{level}_bert_f1"] for r in bert_rows]
    bert_dashboard[level] = round(np.mean(vals), 4)
    print(f"  {level} : {bert_dashboard[level]:.4f}")

bert_dashboard["overall"] = round(np.mean(list(bert_dashboard.values())), 4)
print(f"  Overall : {bert_dashboard['overall']:.4f}")

## 3. G-Eval per-chart-per-level

In [ ]:
import json, time, anthropic
import numpy as np
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

GEVAL_L4_CHART_PROMPT = """You are an expert evaluator for BI dashboard executive insight quality.

You will be given a single L4 sentence — the business implication for ONE chart in a dashboard.

Score the CANDIDATE L4 on these 3 dimensions (1–5 each):
1. Specificity   : Names a specific entity, metric, or chart finding — not generic filler
2. Grounding     : Directly traceable to a visible data pattern — no external assumptions
3. Actionability : Suggests a clear, concrete next step — not vague advice

Respond ONLY with valid JSON:
{{
  "specificity": <1-5>,
  "grounding": <1-5>,
  "actionability": <1-5>,
  "overall": <average rounded to 2 decimal places>,
  "rationale": "<1-2 sentences>"
}}

REFERENCE L4:
{reference}

CANDIDATE L4:
{candidate}
"""

def geval_l4_chart(ai_l4, human_l4):
    msg = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=256,
        messages=[{
            "role": "user",
            "content": GEVAL_L4_CHART_PROMPT.format(
                reference=human_l4,
                candidate=ai_l4
            )
        }]
    )
    raw = msg.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()
    return json.loads(raw)

# ── Score each chart ─────────────────────────────────────────
geval_rows = []

print("G-EVAL L4 PER CHART")
print("=" * 70)
print(f"{'Chart':<35} {'Spec':>5} {'Grnd':>5} {'Actn':>5} {'Ovrl':>5}")
print("-" * 70)

for cid in shared_ids:
    result = geval_l4_chart(ai_ids[cid]["L4"], human_ids[cid]["L4"])
    result["chart_id"] = cid
    result["title"]    = ai_ids[cid]["title"]
    geval_rows.append(result)
    print(f"  {result['title'][:33]:<33} "
          f"  {result['specificity']:>5}"
          f"  {result['grounding']:>5}"
          f"  {result['actionability']:>5}"
          f"  {result['overall']:>5}")
    time.sleep(0.3)  # small delay to avoid rate limiting

# ── Dashboard-level rollup ────────────────────────────────────
print("\nDASHBOARD AVERAGES (G-Eval L4)")
print("-" * 40)
geval_dashboard = {}
for dim in ["specificity", "grounding", "actionability", "overall"]:
    vals = [r[dim] for r in geval_rows]
    geval_dashboard[dim] = round(np.mean(vals), 4)
    print(f"  {dim:<14} : {geval_dashboard[dim]:.4f}")

## Combined results

In [ ]:
import numpy as np

print("EVALUATION SUMMARY — Dashboard 1")
print("=" * 60)

# ── Per-chart table ───────────────────────────────────────────
print(f"\n{'Chart':<35} {'ROUGE-1':^18} {'BERTScore':^18} {'G-Eval L4':^10}")
print(f"{'':35} {'L2':>5} {'L3':>5} {'L4':>5}  {'L2':>5} {'L3':>5} {'L4':>5}  {'Ovrl':>8}")
print("-" * 84)

for i, cid in enumerate(shared_ids):
    title = ai_ids[cid]["title"][:33]
    r = rouge_rows[i]
    b = bert_rows[i]
    g = geval_rows[i]
    print(f"  {title:<33}"
          f"  {r['L2_rouge1']:>5}  {r['L3_rouge1']:>5}  {r['L4_rouge1']:>5}"
          f"  {b['L2_bert_f1']:>5}  {b['L3_bert_f1']:>5}  {b['L4_bert_f1']:>5}"
          f"  {g['overall']:>8.2f}")

# ── Dashboard averages ────────────────────────────────────────
print("-" * 84)
print(f"  {'DASHBOARD AVG':<33}"
      f"  {rouge_dashboard['L2']:>5}  {rouge_dashboard['L3']:>5}  {rouge_dashboard['L4']:>5}"
      f"  {bert_dashboard['L2']:>5}  {bert_dashboard['L3']:>5}  {bert_dashboard['L4']:>5}"
      f"  {geval_dashboard['overall']:>8.4f}")

# ── Level summary ─────────────────────────────────────────────
print(f"\n{'LEVEL SUMMARY':<35} {'ROUGE-1 F1':>10} {'BERTScore F1':>13} {'G-Eval /5':>10}")
print("-" * 70)
for level in ["L2", "L3", "L4"]:
    geval_col = f"{geval_dashboard['overall']:.4f}" if level == "L4" else "—"
    print(f"  {level:<33}"
          f"  {rouge_dashboard[level]:>10}"
          f"  {bert_dashboard[level]:>13}"
          f"  {geval_col:>10}")

print(f"  {'Overall':<33}"
      f"  {rouge_dashboard['overall']:>10}"
      f"  {bert_dashboard['overall']:>13}"
      f"  {geval_dashboard['overall']:>10.4f}")

# ── G-Eval L4 dimension breakdown ────────────────────────────
print(f"\n{'G-EVAL L4 DIMENSIONS':<35} {'Score /5':>8}")
print("-" * 45)
for dim in ["specificity", "grounding", "actionability", "overall"]:
    print(f"  {dim:<33}  {geval_dashboard[dim]:>8.4f}")